# מעבדה 10 — פוטנציאלים תרמודינמיים

במעבדה הזו תיקחו את היחס היסודי של מודול 09 ותקראו אותו דרך ארבעה חלונות — $U$, $H$, $F$
ו-$G$ — שכל אחד מהם מותאם למה שמעבדה יכולה באמת להחזיק קבוע. אחר כך תמדדו שינוי אנטרופיה בעזרת
מד לחץ ומדחום בלבד, ותקראו את האנטרופיה של גומייה מתוחה מתוך מד כוח.

המסלול: מצב אחד וכל פוטנציאל הנבנה ממנו (חלק 1); התמרת לז'נדר, בביצוע נומרי (חלק 2); יחסי
מקסוול כפער בין נגזרות צולבות, וזוג שדות שנכשל במבחן (חלק 3); בוכנה כנגד אמבט חום, שבה האנרגיה
החופשית יורדת בזמן שהאנרגיה עולה (חלק 4); על מה האנתלפיה מנהלת חשבונות, ומה חניקה שומרת (חלק 5);
אנטרופיה ממד לחץ (חלק 6) ועד כמה אפשר לסמוך על המדידה הזו (חלק 7); הגומייה (חלק 8); הבדיקות
האוטומטיות (חלק 9); וחקירה חופשית (חלק 10).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | חומר הנתון על ידי יחס יסודי חלק $S(U, V, N)$ — הגז האידיאלי של סאקור–טטרודה, מוצק איינשטיין, או גז ואן דר ואלס — לבדו, מול אמבט חום ב-$T$, או כשהוא נדחף דרך פקק נקבובי |
| **דינמיקה** | אין עבור הפוטנציאלים, שהם פונקציות מצב; בוכנה משוחררת מועברת דרך שיווי משקל מאולצים בזמן שהאמבט סופג כל חום הנדרש כדי להחזיק את $T$ קבועה |
| **גבול** | דופן מוליכת חום אל האמבט; בוכנה ללא חיכוך במקום שבו לחץ מוחזק; פקק נקבובי מבודד לחניקה |
| **צבר** | לא רלוונטי — האמבט הוא אידאליזציה תרמודינמית, והגרסה הסטטיסטית שלו שייכת למודול 11 |
| **מוזנח** | פלוקטואציות, הגודל הסופי של כל אמבט ממשי, הקצב שבו משהו מגיע לרלקסציה, וכל סוג של עבודה פרט ל-$P\,\mathrm{d}V$ אלא אם נאמר אחרת |
| **תקף כאשר** | האמבט גדול בהרבה מן המערכת, ו-$S$ קעורה ממש במקום שבו משתמשים בה — עבור גז ואן דר ואלס, מעל הטמפרטורה הקריטית שלו |
| **אופני כישלון** | יחס עם שקע, שבו שיפוע אחד מציין כמה מצבים והתמרת לז'נדר מתקפלת (מודול 14); אמבטים קטנים (מודול 11); תהליכים מהירים מכדי לעבור דרך מצבי שיווי משקל |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import fundamental, gases, potentials, processes
from thermolab.constants import AMU, K_B, N_A
from thermolab.validation import convergence_study, relative_error, seed_study

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

ARGON_MASS = 39.948 * AMU
# Argon's van der Waals constants, per particle: a_molar / N_A^2 and b_molar / N_A.
VDW_A, VDW_B = 0.1355 / N_A**2, 3.201e-5 / N_A

argon = fundamental.monatomic_ideal_gas(ARGON_MASS)
argon_vdw = potentials.van_der_waals_gas(ARGON_MASS, VDW_A, VDW_B)
print(f"k_B = {K_B:.6e} J/K")
print(f"van der Waals argon: critical temperature "
      f"{gases.vdw_critical_point(VDW_A, VDW_B)[1]:.1f} K")

## חלק 1 — מצב אחד, כל פוטנציאל

קחו מול אחד של ארגון ב-$300\ \mathrm{K}$ וב-$1\ \mathrm{bar}$. היחס מספק את האנרגיה, האנטרופיה
והלחץ שלו; כל פוטנציאל הוא אז $U$ שנוספו לו או הוחסרו ממנו חלק מן האיברים $TS$ ו-$PV$. מתחת
למספרים, הטבלה שדף המודול בונה: כל פוטנציאל, המשתנים שהוא פונקציה טבעית שלהם, ומתי לפנות אליו.

In [ ]:
n_atoms = N_A
volume = n_atoms * K_B * 300.0 / 1e5
state = potentials.ledger_at(argon, 300.0, volume, n_atoms)

print(f"U                = {state.energy:10.1f} J")
print(f"TS               = {state.ts:10.1f} J")
print(f"PV               = {state.pv:10.1f} J")
print(f"H = U + PV       = {state.enthalpy:10.1f} J")
print(f"F = U - TS       = {state.helmholtz:10.1f} J")
print(f"G = U - TS + PV  = {state.gibbs:10.1f} J")
mu = fundamental.chemical_potential_of(argon, state.energy, volume, n_atoms)
print(f"\nG / N = {state.gibbs / n_atoms:.6e} J;  module 09's slope gives mu = {mu:.6e} J")
print()
for symbol in ("S", "U", "H", "F", "G"):
    nv = potentials.natural_variables(symbol)
    print(f"{nv.symbol}({', '.join(nv.variables)})   {nv.differential:<38}  {nv.behaviour}")

fig, ax = plt.subplots(figsize=(7, 3.2))
names = ["U", "TS", "PV", "H", "F", "G"]
values = [state.energy, state.ts, state.pv, state.enthalpy, state.helmholtz, state.gibbs]
ax.bar(names, np.array(values) / 1e3,
       color=["#111827", "#94a3b8", "#94a3b8", "#d97706", "#2563eb", "#dc2626"])
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("kJ")
ax.set_title("one mole of argon at 300 K and 1 bar")
plt.tight_layout()
plt.show()

שני דברים כדאי לשים לב אליהם. האנרגיות החופשיות גדולות ושליליות, משום ש-$TS$ גדול פי כמה
מ-$U$ עבור גז: רוב מה ש-$TS$ רושם אינו אנרגיה שהגז *מחזיק*, ושום דבר כאן אינו מכל של משהו.
ו-$G/N$ שווה לפוטנציאל הכימי שמודול 09 קרא מתוך השיפוע של היחס — יחס אוילר,
$U = TS - PV + \mu N$, בסידור מחדש, אומר $G = \mu N$.

### לנבא

התחייבו לתשובה לכל אחד מארבעת הניבויים של דף המודול לפני שתמשיכו — הם מה שהמעבדה הזו בודקת:

1. גז בקופסה קשיחה, במגע עם אמבט חום, שאילוץ פנימי שלו משוחרר. האם האנרגיה שלו חייבת לרדת?
2. שני צילינדרים של ארגון בטמפרטורת החדר מחזיקים בדיוק את אותה אנרגיה פנימית. האם אחד מהם
   יכול לבצע יותר עבודה מן השני?
3. מתחו גומייה במהירות ונגעו בה בשפה: חמה יותר, או קרה יותר? אחר כך תלו עליה משקולת וחממו
   אותה במייבש שיער: האם היא נמתחת או מתכווצת?
4. האם $\Delta H$ שווה לחום $Q$ בכל תהליך, או רק בתנאים מסוימים — ואם כן, באילו?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — התמרת לז'נדר, נומרית

$U(S)$ כאשר $V$ ו-$N$ קבועים היא עקומה קמורה. בכל נקודה, לישר המשיק שלה יש שיפוע $T$, והוא
פוגש את הציר $S = 0$ ב-$U - TS = F$. לכן דגימת העקומה, לקיחת השיפועים והחסרה נותנות את $F$
כפונקציה של $T$ — בלי שסופקה כל נוסחה עבור $F$. הצורה הסגורה שהיא אמורה לשחזר,
$F = -N\kB T\,[\ln(V/N\lambda^3) + 1]$ כאשר $\lambda$ הוא אורך הגל התרמי, משמשת רק לבדיקת
התשובה.

In [ ]:
s_mid = state.entropy
s_grid = np.linspace(0.8 * s_mid, 1.2 * s_mid, 10_000)
u_grid = potentials.energy_at_entropy(argon, s_grid, volume, n_atoms)
t_grid, f_grid = potentials.legendre_transform(u_grid, s_grid)


def closed_form_f(t):
    wavelength = fundamental.PLANCK_H / np.sqrt(2 * np.pi * ARGON_MASS * K_B * t)
    return -n_atoms * K_B * t * (np.log(volume / (n_atoms * wavelength**3)) + 1)


fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.plot(s_grid, u_grid / 1e3, color="black")
left.set_xlabel("S (J/K)")
left.set_ylabel("U (kJ)")
left.set_title("U(S) at fixed V and N: convex")
right.plot(t_grid, f_grid / 1e3, color="#2563eb", lw=3, label="tangent intercepts")
right.plot(t_grid, closed_form_f(t_grid) / 1e3, color="black", ls="--", label="closed form")
right.set_xlabel("T (K)")
right.set_ylabel("F (kJ)")
right.legend()
plt.tight_layout()
plt.show()

legendre_error = np.max(np.abs(f_grid - closed_form_f(t_grid)) / np.abs(closed_form_f(t_grid)))
print(f"temperatures covered: {t_grid[0]:.0f} K to {t_grid[-1]:.0f} K")
print(f"largest relative error of F on the curve: {legendre_error:.2e}")

### כמה טובה רשת גסה?

השיפוע של כל משיק בא מהפרש מרכזי, ולכן ה-$T$ של כל דגימה שגוי בסדר שני במרווח הרשת. מדדו שתי
שגיאות שונות ככל שהרשת מתעדנת: כמה רחוקה כל נקודה $(T, F)$ מן העקומה האמיתית *בטמפרטורה שהייתה
אמורה להיות לה*, וכמה רחוקה היא מן העקומה האמיתית *בטמפרטורה שהיא מדווחת*.

In [ ]:
def legendre_errors(n_points):
    s = np.linspace(0.9 * s_mid, 1.1 * s_mid, n_points)
    u = np.asarray(potentials.energy_at_entropy(argon, s, volume, n_atoms))
    slope, intercept = potentials.legendre_transform(u, s)
    exact_t = 2 * u / (3 * n_atoms * K_B)  # the temperature this sample really has
    point = np.max(np.abs(intercept - closed_form_f(exact_t)) / np.abs(closed_form_f(exact_t)))
    curve = np.max(np.abs(intercept - closed_form_f(slope)) / np.abs(closed_form_f(slope)))
    return point, curve


grids = [25, 50, 100, 200, 400]
point_study = convergence_study(lambda k: legendre_errors(k)[0], grids, 0.0)
curve_study = convergence_study(lambda k: legendre_errors(k)[1], grids, 0.0)
for k, p_err, c_err in zip(grids, point_study.errors, curve_study.errors, strict=True):
    print(f"{k:4d} points:  off at its own T {p_err:.2e}    off the curve {c_err:.2e}")
print(f"\nobserved order: each point {point_study.observed_order:.2f},  "
      f"the curve {curve_study.observed_order:.2f}")

כל נקודה ממוקמת שלא במקומה בסדר שני, כפי שמצופה מהפרש מרכזי. אבל הנקודות מחליקות *לאורך*
העקומה האמיתית ולא אל מחוצה לה, והעקומה שהן משרטטות נכונה בסדר רביעי. הסיבה היא אותה סיבה
שבגללה ההתמרה עובדת בכלל: נקודת החיתוך $f - px$ סטציונרית ב-$x$ בנקודת ההשקה האמיתית, ולכן
שגיאה בשיפוע מזיזה את הנקודה לאורך $F(T)$ ורק הריבוע של השגיאה מזיז אותה ממנה.

## חלק 3 — יחסי מקסוול כפער בין נגזרות צולבות

$\mathrm{d}F = -S\,\mathrm{d}T - P\,\mathrm{d}V$ מדויק, ולכן המבחן של מודול 05 אומר
$(\partial S/\partial V)_T = (\partial P/\partial T)_V$. על היחס של ואן דר ואלס, $S$ הוא *ערך*
של היחס במצב הממוקם ו-$P$ הוא *יחס בין השיפועים שלו*: שתי פעולות שונות, שיחס מקסוול אומר שחייבות
להסכים. `maxwell_check` מחזירה את הפער ביחס לגודל שתי הנגזרות הצולבות, בכל נקודה של רשת.

In [ ]:
t_axis = np.linspace(200.0, 600.0, 60)
v_axis = np.geomspace(1.5e-4, 5e-3, 60)
tt, vv = np.meshgrid(t_axis, v_axis)
minus_s, minus_p = potentials.helmholtz_form(argon_vdw, N_A)

steps = [4e-2, 2e-2, 1e-2, 5e-3]
gaps = [np.abs(potentials.maxwell_check(minus_s, minus_p, tt, vv, rel_step=h)) for h in steps]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, h, gap in zip(axes, steps, gaps, strict=True):
    image = ax.pcolormesh(t_axis, v_axis * 1e3, gap, norm="log", vmin=1e-6, vmax=1e-3,
                          cmap="magma", shading="auto")
    ax.set_yscale("log")
    ax.set_title(f"h = {h:g}")
    ax.set_xlabel("T (K)")
axes[0].set_ylabel("V (L)")
fig.colorbar(image, ax=axes, label="|gap|")
plt.show()

gap_study = convergence_study(
    lambda k: float(np.max(np.abs(potentials.maxwell_check(minus_s, minus_p, tt, vv,
                                                           rel_step=1.0 / k)))),
    [25, 50, 100, 200], 0.0)
for k, err in zip(gap_study.refinements, gap_study.errors, strict=True):
    print(f"h = {1 / k:.4f}   largest |gap| = {err:.3e}")
print(f"\nobserved order = {gap_study.observed_order:.2f}   (central differences: 2)")

בדיקה שיכולה רק לעבור אינה בדיקה. זווגו את האנטרופיה של הגז *האידיאלי* עם הלחץ של *ואן דר ואלס*
— שני חומרים המעמידים פנים שהם אחד — והריצו את אותו מבחן.

In [ ]:
minus_s_ideal, _ = potentials.helmholtz_form(argon, N_A)
nb = N_A * VDW_B
print("   V (L)     gap at h=1e-2   gap at h=1e-3    Nb / (2V - Nb)")
for v in (1.5e-4, 3e-4, 1e-3, 5e-3):
    coarse = potentials.maxwell_check(minus_s_ideal, minus_p, 300.0, v, rel_step=1e-2)
    fine = potentials.maxwell_check(minus_s_ideal, minus_p, 300.0, v, rel_step=1e-3)
    print(f"{v * 1e3:8.2f}    {float(coarse):12.6f}   {float(fine):12.6f}   "
          f"{nb / (2 * v - nb):12.6f}")

עבור יחס אמיתי הפער הוא רק שגיאת הקיטוע של ההפרשים: הוא יורד פי ארבעה עם כל חציה של הצעד, עד
רצפה של כ-$10^{-7}$ הנקבעת על ידי שגיאת עיגול. הפער של הזוג שאינו מתאים אינו זז כשהצעד מתעדן,
והוא יושב בדיוק על $Nb/(2V - Nb)$ — האנטרופיה האידיאלית גדלה כ-$N\kB/V$ עם הנפח, בעוד שהלחץ
של ואן דר ואלס עולה כ-$N\kB/(V - Nb)$ עם הטמפרטורה. המבחן אומר לכם לא רק *ששני* השדות באו
מחומרים שונים, אלא גם בכמה.

## חלק 4 — בוכנה כנגד אמבט חום

### לנבא

קופסה קשיחה, במגע עם אמבט ב-$300\ \mathrm{K}$, מחזיקה בוכנה. בצד A יש שני מולים של ארגון ברבע
מן הקופסה; בצד B יש מול אחד בכל השאר. הבוכנה משוחררת ונסחפת, בבלימה, עד שהיא נעצרת. האם
האנרגיה הכוללת של הארגון עולה, יורדת או נשארת זהה? ומה לגבי האנרגיה החופשית שלו?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
trace = potentials.free_energy_minimization(argon_vdw, 300.0, 4e-3, 2 * N_A, N_A,
                                            start_share=0.25, n_steps=201)
d_u = trace.energy - trace.energy[0]
d_f = trace.free_energy - trace.free_energy[0]
t_ds_system = 300.0 * trace.system_entropy_change
t_ds_bath = 300.0 * trace.bath_entropy_change
t_ds_total = 300.0 * trace.total_entropy_change

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.plot(trace.share, d_u, color="#dc2626", label="Delta U")
left.plot(trace.share, d_f, color="#2563eb", label="Delta F")
left.set_xlabel("V_A / V")
left.set_ylabel("J")
left.legend()
left.set_title("the argon")
right.plot(trace.share, t_ds_system, color="#111827", label="T Delta S (argon)")
right.plot(trace.share, t_ds_bath, color="#94a3b8", label="T Delta S (bath)")
right.plot(trace.share, t_ds_total, color="#d97706", lw=2.5, label="T Delta S (total)")
right.set_xlabel("V_A / V")
right.legend()
right.set_title("the entropy books, times T")
plt.tight_layout()
plt.show()

print(f"piston stops at V_A/V = {trace.share[-1]:.6f}   (equal densities: {2 / 3:.6f})")
print(f"pressures there: {trace.pressure_a[-1] / 1e5:.4f} bar | "
      f"{trace.pressure_b[-1] / 1e5:.4f} bar")
print(f"Delta U             = {d_u[-1]:+9.1f} J")
print(f"Delta F             = {d_f[-1]:+9.1f} J")
print(f"T Delta S (argon)   = {t_ds_system[-1]:+9.1f} J")
print(f"T Delta S (bath)    = {t_ds_bath[-1]:+9.1f} J")
print(f"Delta S (total)     = {trace.total_entropy_change[-1]:+9.4f} J/K")

ideal = potentials.free_energy_minimization(argon, 300.0, 4e-3, 2 * N_A, N_A,
                                            start_share=0.25, n_steps=21)
print(f"\nideal argon instead: Delta U = {ideal.energy[-1] - ideal.energy[0]:+.2e} J,  "
      f"Delta F = {ideal.free_energy[-1] - ideal.free_energy[0]:+.1f} J")

האנרגיה של הארגון *עלתה* — בשיעור שהמשיכה ההדדית של האטומים שלו איבדה כשהצד הדחוס התפשט — והארגון
התייצב במצב בעל האנרגיה **הגדולה ביותר** שהבוכנה יכלה להגיע אליו. האנרגיה החופשית שלו ירדה אל
הקטנה ביותר. אין כאן פרדוקס: האמבט שילם עבור העלייה, והאנטרופיה של האמבט ירדה ב-$\Delta U / T$,
אבל האנטרופיה של הארגון עלתה בהרבה יותר. הסכום, ארגון ואמבט, טיפס לאורך כל הדרך, ו-$-\Delta F / T$
הוא בדיוק הסכום הזה. כאשר $T$ ו-$V$ קבועים, "למזער את $F$" *הוא* "למקסם את האנטרופיה של הכול",
כשהוא נאמר על המערכת לבדה.

לגז האידיאלי אין משיכה, ולכן האנרגיה שלו בטמפרטורה קבועה אינה יכולה להשתנות כלל; האנרגיה
החופשית שלו יורדת בכל זאת. אף אחד מן הגזים לא מיזער את האנרגיה שלו.

### אותה אנרגיה, תועלת שונה

שני צילינדרים מחזיקים כל אחד מול אחד של ארגון ב-$300\ \mathrm{K}$: אחד ב-$1\ \mathrm{L}$, ואחד
ב-$10\ \mathrm{L}$. האנרגיות שלהם שוות. בחדר ב-$300\ \mathrm{K}$, העבודה המרבית שכל אחד מהם
יכול לספק היא האנרגיה החופשית שלו פחות זו של המצב שבו הוא יסיים.

In [ ]:
small = potentials.ledger_at(argon, 300.0, 1e-3, N_A)
large = potentials.ledger_at(argon, 300.0, 10e-3, N_A)
print(f"U:  {small.energy:.1f} J  vs  {large.energy:.1f} J")
print(f"F:  {small.helmholtz:.1f} J  vs  {large.helmholtz:.1f} J")
print(f"extra work the compressed cylinder can deliver at 300 K: "
      f"{small.helmholtz - large.helmholtz:.1f} J   (R T ln 10 = "
      f"{N_A * K_B * 300.0 * np.log(10):.1f} J)")

## חלק 5 — על מה האנתלפיה מנהלת חשבונות

**חימום בלחץ קבוע.** התהליך האיזוברי של מודול 06 מחשב את החום שלו כ-$C_P\,\Delta T$. השוו אותו
לשינוי של $H = U + PV$ בין שני מצבי הקצה, הממוקמים על משטח האנטרופיה.

In [ ]:
start = processes.EquilibriumState.from_temperature(10**22, 300.0, 4e-4)
heating = processes.isobaric(start, 6e-4)
h_start, h_end = (potentials.ledger_at(argon, s.temperature, s.volume, 10**22).enthalpy
                  for s in (heating.start, heating.end))
print(f"Q_P from C_P Delta T      = {heating.heat:.6f} J")
print(f"Delta H from the relation = {h_end - h_start:.6f} J")
print(f"work done ON the gas      = {heating.work_on_gas:.6f} J"
      "   (the -P Delta V the heat also pays)")

# Splitting water at 298.15 K and 1 bar, per mole, from standard tables:
delta_h, delta_g = 285.83e3, 237.13e3  # J/mol
faraday = 96485.33
print("\nelectrolysis of one mole of water, run reversibly:")
print(f"  electrical work done on the cell = Delta G = {delta_g / 1e3:.2f} kJ"
      f"   (cell voltage {delta_g / (2 * faraday):.3f} V)")
print(f"  heat drawn from the surroundings = Delta H - W_elec = {(delta_h - delta_g) / 1e3:.2f} kJ")
print(f"  Delta H                          = {delta_h / 1e3:.2f} kJ  -- not the heat")

**חניקה.** דחפו גז בזרימה יציבה דרך פקק נקבובי מ-$50\ \mathrm{bar}$ ל-$1\ \mathrm{bar}$, בבידוד.
האנתלפיה זהה משני הצדדים; שום דבר אחר אינו חייב להיות זהה.

In [ ]:
for label, relation in (("ideal argon", argon), ("van der Waals argon", argon_vdw)):
    out = potentials.throttle(relation, N_A, 300.0, 50e5, 1e5)
    print(f"{label:<20} T: {out.temperature_in:.2f} -> {out.temperature_out:.2f} K   "
          f"U: {out.energy_in:8.1f} -> {out.energy_out:8.1f} J   "
          f"H: {out.enthalpy_in:8.1f} -> {out.enthalpy_out:8.1f} J")

small_drop = potentials.throttle(argon_vdw, N_A, 300.0, 1.1e5, 1.0e5)
coefficient = small_drop.temperature_change / (1.0e5 - 1.1e5)
estimate = (2 * VDW_A / (K_B * 300.0) - VDW_B) / (2.5 * K_B)
print(f"\nJoule-Thomson coefficient at 1 bar: {coefficient * 1e5:.4f} K/bar  "
      f"(dilute van der Waals formula: {estimate * 1e5:.4f} K/bar)")

הגז האידיאלי יוצא מן הפקק בטמפרטורה שבה נכנס: האנתלפיה שלו תלויה ב-$T$ בלבד. גז ואן דר ואלס
מתקרר בכ-$18\ \mathrm{K}$ — בטמפרטורת החדר המשיכה בין האטומים שלו גוברת על הגודל שלהם, והרחקתם
זה מזה עולה באנרגיה שרק התנועה שלהם יכולה לספק. כך מנזילים חנקן ואוויר. בשני המקרים $H$ נשמרת
במדויק ו-$U$ אינה נשמרת.

## חלק 6 — אנטרופיה ממד לחץ

### לנבא

יש לכם מד לחץ הקורא לחץ בדיוק של $0.2\%$ ומדחום. אתם מודדים את הלחץ של גז בתשע טמפרטורות, בכל
אחד מתשעה נפחים בין $1$ ל-$2\ \mathrm{L}$. האם תוכלו לקבוע את שינוי האנטרופיה שלו בהכפלת נפחו
בטמפרטורה קבועה — ובאיזה דיוק?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
temperatures = np.linspace(280.0, 320.0, 9)
volumes = 1e-3 * np.geomspace(1.0, 2.0, 9)
n_gas = 1e21
readings = potentials.gauge_readings(argon, n_gas, temperatures, volumes, 0.002, rng)
result = potentials.entropy_from_gauge(temperatures, volumes, readings)
exact = n_gas * K_B * np.log(2)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
colours = plt.cm.viridis(np.linspace(0, 1, volumes.size))
for row, colour in zip(readings, colours, strict=True):
    left.plot(temperatures, row / 1e3, "o", color=colour, ms=4)
    fit = np.polyfit(temperatures, row, 1)
    left.plot(temperatures, np.polyval(fit, temperatures) / 1e3, color=colour, lw=1)
left.set_xlabel("T (K)")
left.set_ylabel("P (kPa)")
left.set_title("the gauge: one straight line per volume")
right.errorbar(volumes * 1e3, result.slopes * volumes, yerr=result.slope_errors * volumes,
               fmt="o", color="black")
right.axhline(n_gas * K_B, color="#dc2626", ls="--", label="N k_B")
right.set_xlabel("V (L)")
right.set_ylabel("V (dP/dT)_V  (J/K)")
right.legend()
right.set_title("the integrand, from the fitted slopes")
plt.tight_layout()
plt.show()

print(f"Delta S from gauge and thermometer = {result.entropy_change[-1]:.5e} "
      f"+/- {result.entropy_error[-1]:.1e} J/K")
print(f"N k_B ln 2                        = {exact:.5e} J/K")
print(f"difference, in error bars          = "
      f"{(result.entropy_change[-1] - exact) / result.entropy_error[-1]:+.2f}")
print(f"relative uncertainty               = {result.entropy_error[-1] / exact:.2%}")

אותה מדידה על גז ואן דר ואלס דחוס: מול אחד, מ-$0.2$ עד $0.4\ \mathrm{L}$.

In [ ]:
v_dense = np.geomspace(2e-4, 4e-4, 65)
readings_vdw = potentials.gauge_readings(argon_vdw, N_A, temperatures, v_dense, 0.0, rng)
dense = potentials.entropy_from_gauge(temperatures, v_dense, readings_vdw)
nb = N_A * VDW_B
print(f"Delta S from the gauge          = {dense.entropy_change[-1]:.5f} J/K")
print(f"N k_B ln((V2 - Nb)/(V1 - Nb))   = {N_A * K_B * np.log((4e-4 - nb) / (2e-4 - nb)):.5f} J/K")
print(f"N k_B ln 2 (if it were ideal)   = {N_A * K_B * np.log(2):.5f} J/K")

אף אחת מן המדידות לא כללה דבר הקורא אנטרופיה. יחס מקסוול הפך שיפוע של לחץ כנגד טמפרטורה לשיפוע
של אנטרופיה כנגד נפח. עבור הגז הדחוס המשיכה נופלת לחלוטין — בנפח קבוע היא מורידה את הלחץ באותה
מידה בכל טמפרטורה, ולכן אינה נוגעת ב-$(\partial P/\partial T)_V$ — בעוד שהנפח המודר אינו נופל:
לאטומים יש פחות מקום להיות בו, והאנטרופיה שלהם גדלה יותר כשהקופסה גדלה.

## חלק 7 — האם אפשר לסמוך על פס השגיאה?

ריצה אחת היא אנקדוטה. חזרו על מדידת מד הלחץ תחת עשרים זרעים בלתי תלויים: הממוצע אמור לשבת על
$N\kB \ln 2$, והפיזור בין הריצות אמור להתאים לפס השגיאה שכל ריצה בודדת מדווחת על עצמה.

In [ ]:
def gauge_delta_s(generator):
    noisy = potentials.gauge_readings(argon, n_gas, temperatures, volumes, 0.002, generator)
    return float(potentials.entropy_from_gauge(temperatures, volumes, noisy).entropy_change[-1])


study = seed_study(gauge_delta_s, n_seeds=20)
scatter = float(np.std(study.values, ddof=1))
print(f"mean over 20 seeds = {study.mean:.5e} +/- {study.standard_error:.1e} J/K   "
      f"(N k_B ln 2 = {exact:.5e})")
print(f"scatter between runs      = {scatter:.2e} J/K")
print(f"error bar a single run reports = {result.entropy_error[-1]:.2e} J/K")

## חלק 8 — הגומייה

על גומייה הנמשכת במתיחות $f$ נעשית עבודה $f\,\mathrm{d}L$ בזמן שהיא נמתחת, ולכן האנרגיה החופשית
שלה מקיימת $\mathrm{d}F = -S\,\mathrm{d}T + f\,\mathrm{d}L$, ויחס מקסוול שלה הוא

$$
\left(\frac{\partial S}{\partial L}\right)_T = -\left(\frac{\partial f}{\partial T}\right)_L .
$$

מד כוח ומדחום מודדים את האגף הימני. `data/10-rubber-band.csv` הוא ריצה באורך קבוע (קראו את
הכותרת שלו כדי לדעת בדיוק מה הוא): סריקת חימום, ואחריה סריקת קירור.

### לנבא

לפני הרצת התא הבא: האם המתיחות באורך קבוע תעלה או תרד כשהגומייה מתחממת? ומה אומרת התשובה שלכם
על הסימן של $(\partial S/\partial L)_T$ — האם מתיחת גומייה מעלה את האנטרופיה שלה או מורידה אותה?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "10-rubber-band.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "10-rubber-band.csv")

band = np.genfromtxt(csv_path, delimiter=",", comments="#")
band_t, band_f, sweep = band.T
warming, cooling = sweep == 1, sweep == 2


def fit(t, f):
    coeffs, cov = np.polyfit(t, f, 1, cov=True)
    return coeffs, float(np.sqrt(cov[0, 0]))


(slope, intercept), slope_error = fit(band_t, band_f)
(slope_h, _), error_h = fit(band_t[warming], band_f[warming])
(slope_c, _), error_c = fit(band_t[cooling], band_f[cooling])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(band_t[warming], band_f[warming], "o", color="#dc2626", label="heating")
ax.plot(band_t[cooling], band_f[cooling], "s", color="#2563eb", label="cooling")
line_t = np.linspace(band_t.min(), band_t.max(), 50)
ax.plot(line_t, intercept + slope * line_t, color="black", lw=1)
ax.set_xlabel("T (K)")
ax.set_ylabel("tension f (N)")
ax.legend()
plt.tight_layout()
plt.show()

t_room = 295.0
f_room = intercept + slope * t_room
print(f"(df/dT)_L, all readings  = {slope * 1e3:.2f} +/- {slope_error * 1e3:.2f} mN/K")
print(f"            heating only = {slope_h * 1e3:.2f} +/- {error_h * 1e3:.2f} mN/K")
print(f"            cooling only = {slope_c * 1e3:.2f} +/- {error_c * 1e3:.2f} mN/K")
print(f"(dS/dL)_T = -(df/dT)_L   = {-slope * 1e3:.2f} mJ/(K m)")
print(f"\nat {t_room:.0f} K the tension is {f_room:.3f} N, of which T (df/dT)_L = "
      f"{t_room * slope:.3f} N is entropic")
print(f"energetic share of the tension: {1 - t_room * slope / f_room:.1%}")

המתיחות עולה עם הטמפרטורה, ולכן מתיחת גומייה בטמפרטורה קבועה *מורידה* את האנטרופיה שלה:
המולקולות הארוכות נמשכות מתוך הסידורים המסובכים, רבי ריבוי המצבים, שלהן. כל השאר נובע מן הסימן
האחד הזה.

- **השפה.** מתחו את הגומייה מהר ושום חום אינו מספיק לצאת, ולכן האנטרופיה שלה נשארת קבועה.
  לשרשראות יש פחות סידורים זמינים, ולכן האנטרופיה חייבת להימצא במקום אחר: ברטט מהיר יותר.
  הגומייה מתחממת.
- **המשקולת.** החזיקו את המתיחות קבועה וחממו את הגומייה: המתיחות שהייתה לה באורך הזה עולה,
  ולכן היא מושכת את המשקולת מעלה. גומייה עמוסה מתכווצת בחימום — ההפך מכמעט כל חומר אחר.
- **החלק האנרגטי.** רק כ-$15\%$ מן המתיחות הוא האנרגיה של קשרים מתוחים. השאר הוא אנטרופיה: הכוח
  האנטרופי של מודול 08, נמדד.

סריקות החימום והקירור נותנות שיפועים הנבדלים בכמעט פעמיים השגיאה המשולבת שלהם, וסריקת הקירור
יושבת מעט נמוך יותר: גומי מתוח מתרפה לאט, ולכן הדגימה לא הייתה בדיוק אותו חומר בשתי הסריקות.
התאמת כל הקריאות יחד ממצעת על פני זה.

## חלק 9 — בדיקות אוטומטיות

סימולציה שלא בדקתם היא תמונה, לא ראיה. אלה אותן טענות הרצות בחבילת הבדיקות של הפרויקט.

In [ ]:
# 1. The ledger closes, and G = mu N (Part 1).
assert relative_error(state.energy, state.helmholtz + state.ts) < 1e-12
assert relative_error(state.gibbs / n_atoms, mu) < 1e-6

# 2. The numerical Legendre transform lies on the closed-form F(T) (Part 2) --
#    the curve at fourth order, each point at second.
assert legendre_error < 1e-6
assert 3.7 < curve_study.observed_order < 4.5
assert 1.8 < point_study.observed_order < 2.2

# 3. The Maxwell gap of a genuine relation falls at second order (Part 3) ...
assert 1.9 < gap_study.observed_order < 2.1
# ... and a mismatched pair's does not fall at all.
fake = potentials.maxwell_check(minus_s_ideal, minus_p, 300.0, 3e-4, rel_step=1e-3)
assert relative_error(float(fake), nb / (2 * 3e-4 - nb)) < 1e-3

# 4. The piston: F falls, total entropy climbs, U climbs; it stops at equal pressures (Part 4).
assert np.all(np.diff(trace.free_energy) <= 0)
assert np.all(np.diff(trace.total_entropy_change) >= 0)
assert d_u[-1] > 0
assert relative_error(trace.pressure_a[-1], trace.pressure_b[-1]) < 1e-6

# 5. Q_P = Delta H, and a throttle conserves H but cools only the real gas (Part 5).
assert relative_error(h_end - h_start, heating.heat) < 1e-8
assert relative_error(potentials.throttle(argon, N_A, 300.0, 50e5, 1e5).temperature_out,
                      300.0) < 1e-9
assert relative_error(coefficient, estimate) < 5e-3

# 6. The gauge measures N k_B ln 2, and its error bar is honest (Parts 6 and 7).
assert abs(result.entropy_change[-1] - exact) < 3.5 * result.entropy_error[-1]
assert study.agrees_with(exact, n_sigma=3.5)
assert 0.6 < scatter / result.entropy_error[-1] < 1.6

# 7. The rubber band's entropy falls as it is stretched (Part 8).
assert slope > 0 and slope > 5 * slope_error
print("all checks passed")

## חלק 10 — חקרו בעצמכם

בחרו את הגז, את טמפרטורת האמבט, את חלוקת החלקיקים ואת המקום שבו הבוכנה מתחילה, ואז לחצו על
**Run Interact**. הלוח השמאלי מראה את האנרגיה והאנרגיה החופשית של הגז בזמן שהבוכנה נעה; הלוח
הימני מראה את מאזני האנטרופיה. שלושה דברים כדאי לנסות:

1. עברו לגז האידיאלי. מה קורה ל-$\Delta U$?
2. שימו אותו מספר חלקיקים בשני הצדדים, והתחילו את הבוכנה רחוק לצד אחד.
3. הורידו את הטמפרטורה לעבר הטמפרטורה הקריטית של ואן דר ואלס, כ-$151\ \mathrm{K}$. האם
   $\Delta U$ גדל או קטן ביחס ל-$\Delta F$?

In [ ]:
import ipywidgets as widgets


def explore(gas="van der Waals", temperature=300.0, moles_a=2.0, moles_b=1.0, start_share=0.25):
    relation = argon_vdw if gas == "van der Waals" else argon
    run = potentials.free_energy_minimization(relation, temperature, 4e-3, moles_a * N_A,
                                              moles_b * N_A, start_share, n_steps=121)
    du = run.energy - run.energy[0]
    df = run.free_energy - run.free_energy[0]
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.plot(run.share, du, color="#dc2626", label="Delta U")
    left.plot(run.share, df, color="#2563eb", label="Delta F")
    left.set_xlabel("V_A / V")
    left.set_ylabel("J")
    left.legend()
    right.plot(run.share, temperature * run.system_entropy_change, color="#111827",
               label="T Delta S (gas)")
    right.plot(run.share, temperature * run.bath_entropy_change, color="#94a3b8",
               label="T Delta S (bath)")
    right.plot(run.share, temperature * run.total_entropy_change, color="#d97706", lw=2.5,
               label="T Delta S (total)")
    right.set_xlabel("V_A / V")
    right.legend()
    plt.tight_layout()
    plt.show()
    print(f"piston stops at V_A/V = {run.share[-1]:.4f};  Delta U = {du[-1]:+.1f} J,  "
          f"Delta F = {df[-1]:+.1f} J,  Delta S_total = {run.total_entropy_change[-1]:+.4f} J/K")


widgets.interact_manual(
    explore,
    gas=["van der Waals", "ideal"],
    temperature=widgets.FloatSlider(value=300.0, min=160.0, max=600.0, step=10.0),
    moles_a=widgets.FloatSlider(value=2.0, min=0.2, max=4.0, step=0.1),
    moles_b=widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1),
    start_share=widgets.FloatSlider(value=0.25, min=0.05, max=0.95, step=0.01),
);

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון הנבדק אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "10-potentials.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

רשמו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בנימוקכם?
2. בחלק 4 הארגון סיים באנרגיה *הגבוהה ביותר* שהבוכנה יכלה להגיע אליה. הסבירו, במונחים של
   האמבט, מדוע זה לא הפר שום דבר.
3. מד הלחץ בחלק 6 מעולם לא מדד אנטרופיה. אמרו בדיוק איזה צעד הפך קריאות של לחץ וטמפרטורה
   לכזו.
4. סטודנט אומר שהמתיחות של הגומייה היא "בדיוק כמו של קפיץ". השתמשו במספרים שלכם מחלק 8 כדי
   לומר מה נכון ומה שגוי בכך.

**התשובות שלכם:**

1.
2.
3.
4.